In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

save_path = "../data/"

if not os.path.exists(save_path + "principal_components/"):
        os.makedirs(save_path + "principal_components/")
        
def PCA_transform(matrix):
    pca = PCA(n_components = 2)
    pca.fit(matrix)
    transformed = pca.transform(matrix)
    return transformed, np.around(pca.explained_variance_ratio_*100,1)

In [ ]:
datasets = ["semcor&omsti_noun-synsets-strat", "semcor&omsti_person.n.01", "cctweets-activist", "cctweets-random"]
models = ["bert-base-uncased", "microsoft-deberta-base"]


for dataset in datasets:
    
    try:
        dataset_name, filter_name = dataset.split("_")
    except:
        dataset_name = dataset
        filter_name = None
    
    df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)
    
    filter_mask = np.arange(0, len(df)) if filter_name is None else np.where(df[filter_name])
    
    embedding_files = [file for file in os.listdir(save_path + "embedding") if dataset_name in file and file.endswith(".npy")]
    embedding_files.sort()
    
    embedding_files = [file for file in embedding_files if file.split("_")[1] in models]
    
    
    for embedding_file in embedding_files:
    
        embeddings = np.load(f"{save_path}embedding/{embedding_file}")

        assert embeddings.shape[0] == len(df), "Number of vectors not equal to df length"
    
        embeddings = embeddings[filter_mask]
        
        pca, ev = PCA_transform(embeddings)
        
        
        embedding_name = f"{embedding_file.split('.')[0]}_{filter_name}"
        
        with open(f"{save_path}principal_components/{embedding_name}_pc1-ev{round(ev[0],2)}.npy", "wb") as f:
        
                np.save(f, pca[:,0])
            
        with open(f"{save_path}principal_components/{embedding_name}_pc2-ev{round(ev[1],2)}.npy", "wb") as f:
        
                np.save(f, pca[:,1])
            
            
            
        
        